<a href="https://colab.research.google.com/github/wanchenlang-max/econ5200-lab/blob/lab13/lab13.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [30]:
import pandas as pd
import statsmodels.formula.api as smf
import matplotlib.pyplot as plt

df = pd.read_csv('Zillow_California_2026_Hedonic.csv')
df.head()



,Property_Age,Distance_to_Tech_Hub,Sale_Price
0,77.5,38.1,684100.56
1,11.0,95.1,413634.22
2,47.7,73.5,456709.35
3,61.9,60.3,624533.95
4,100.8,16.4,870137.54


In [5]:
naive_model = smf.ols('Sale_Price ~ Property_Age', data=df).fit()

In [6]:
print(naive_model.summary())

                            OLS Regression Results                            
Dep. Variable:             Sale_Price   R-squared:                       0.757
Model:                            OLS   Adj. R-squared:                  0.757
Method:                 Least Squares   F-statistic:                     3105.
Date:                Fri, 13 Mar 2026   Prob (F-statistic):          1.26e-308
Time:                        17:45:36   Log-Likelihood:                -12818.
No. Observations:                1000   AIC:                         2.564e+04
Df Residuals:                     998   BIC:                         2.565e+04
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
                   coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------
Intercept     3.013e+05   7218.570     41.742   

In [7]:
print("Naive Age Coefficient:", naive_model.params['Property_Age'])

Naive Age Coefficient: 5573.630595439929


In [8]:
if naive_model.params['Property_Age'] < 0:
    print("The coefficient for 'Property_Age' is negative, indicating that in the naive model, the older the property, the lower the price")
else:
    print("The coefficient for 'Property_Age' is positive, indicating that in the naive model, the older the property, the higher the price。")

The coefficient for 'Property_Age' is positive, indicating that in the naive model, the older the property, the higher the price。


In [9]:
multi_model = smf.ols('Sale_Price ~ Property_Age + Distance_to_Tech_Hub', data=df).fit()

In [10]:
print(multi_model.summary())

                            OLS Regression Results                            
Dep. Variable:             Sale_Price   R-squared:                       0.954
Model:                            OLS   Adj. R-squared:                  0.954
Method:                 Least Squares   F-statistic:                 1.040e+04
Date:                Fri, 13 Mar 2026   Prob (F-statistic):               0.00
Time:                        17:48:53   Log-Likelihood:                -11982.
No. Observations:                1000   AIC:                         2.397e+04
Df Residuals:                     997   BIC:                         2.399e+04
Df Model:                           2                                         
Covariance Type:            nonrobust                                         
                           coef    std err          t      P>|t|      [0.025      0.975]
----------------------------------------------------------------------------------------
Intercept             1.203e+06 

In [11]:
print("Multivariate Age Coefficient:", multi_model.params['Property_Age'])

Multivariate Age Coefficient: -2063.1292168021246


In [12]:
print("Step 1 coefficient:", naive_model.params['Property_Age'])
print("Step 2 coefficient:", multi_model.params['Property_Age'])

Step 1 coefficient: 5573.630595439929
Step 2 coefficient: -2063.1292168021246


In [14]:
res_y_model = smf.ols('Sale_Price ~ Distance_to_Tech_Hub', data=df).fit()
df['Price_Residuals'] = res_y_model.resid

In [15]:
df[['Sale_Price', 'Distance_to_Tech_Hub', 'Price_Residuals']].head()

,Sale_Price,Distance_to_Tech_Hub,Price_Residuals
0,684100.56,38.1,-56823.332740
1,413634.22,95.1,19000.990249
2,456709.35,73.5,-69149.815200
3,624533.95,60.3,18481.157582
4,870137.54,16.4,-2619.815668


In [17]:
res_x_model = smf.ols('Property_Age ~ Distance_to_Tech_Hub', data=df).fit()
df['Age_Residuals'] = res_x_model.resid

In [18]:
df[['Property_Age', 'Distance_to_Tech_Hub', 'Age_Residuals']].head()

,Property_Age,Distance_to_Tech_Hub,Age_Residuals
0,77.5,38.1,0.621803
1,11.0,95.1,-13.689856
2,47.7,73.5,3.233510
3,61.9,60.3,5.347789
4,100.8,16.4,4.053610


In [19]:
fwl_model = smf.ols('Price_Residuals ~ Age_Residuals - 1', data=df).fit()

In [20]:
print(fwl_model.summary())

                                 OLS Regression Results                                
Dep. Variable:        Price_Residuals   R-squared (uncentered):                   0.217
Model:                            OLS   Adj. R-squared (uncentered):              0.216
Method:                 Least Squares   F-statistic:                              276.5
Date:                Fri, 13 Mar 2026   Prob (F-statistic):                    5.27e-55
Time:                        17:53:19   Log-Likelihood:                         -11982.
No. Observations:                1000   AIC:                                  2.397e+04
Df Residuals:                     999   BIC:                                  2.397e+04
Df Model:                           1                                                  
Covariance Type:            nonrobust                                                  
                    coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------

In [21]:
print("FWL Isolated Age Coefficient:", fwl_model.params['Age_Residuals'])

FWL Isolated Age Coefficient: -2063.129216802139


In [22]:
print("Step 2 Property_Age coefficient:", multi_model.params['Property_Age'])
print("Step 3 FWL coefficient:", fwl_model.params['Age_Residuals'])

Step 2 Property_Age coefficient: -2063.1292168021246
Step 3 FWL coefficient: -2063.129216802139


In [23]:
print("Difference:", multi_model.params['Property_Age'] - fwl_model.params['Age_Residuals'])

Difference: 1.4551915228366852e-11


In [24]:
print("Step 2 rounded:", round(multi_model.params['Property_Age'], 6))
print("Step 3 rounded:", round(fwl_model.params['Age_Residuals'], 6))

Step 2 rounded: -2063.129217
Step 3 rounded: -2063.129217


In [31]:
# ============================================================
# 3D Hyperplane Visualization for Multivariate OLS Regression
# Predicting Sale_Price ~ Property_Age + Distance_to_Tech_Hub
# ============================================================

import numpy as np
import pandas as pd
import statsmodels.api as sm
import plotly.graph_objects as go

# ----------------------------------------------------------
# STEP 1: 生成示例数据（如果你已有真实数据，替换这部分）
# ----------------------------------------------------------
np.random.seed(42)
n = 200

Property_Age        = np.random.uniform(1, 50, n)       # 房龄（年）
Distance_to_Tech_Hub = np.random.uniform(1, 30, n)      # 距科技中心距离（公里）

# 真实关系：价格随房龄下降、随距离下降，加入噪声
Sale_Price = (
    800000
    - 5000  * Property_Age
    - 12000 * Distance_to_Tech_Hub
    + np.random.normal(0, 30000, n)
)

df = pd.DataFrame({
    "Sale_Price":          Sale_Price,
    "Property_Age":        Property_Age,
    "Distance_to_Tech_Hub": Distance_to_Tech_Hub
})

# ----------------------------------------------------------
# STEP 2: 用 statsmodels 拟合多元 OLS 回归
# ----------------------------------------------------------
X = df[["Property_Age", "Distance_to_Tech_Hub"]]
X = sm.add_constant(X)          # 添加截距项（常数列）
y = df["Sale_Price"]

model = sm.OLS(y, X).fit()
print(model.summary())

# ----------------------------------------------------------
# STEP 3: 提取回归系数
# 模型结果对象 model.params 是一个 Series，索引对应列名
# model.params["const"]                  → 截距 β₀
# model.params["Property_Age"]           → β₁
# model.params["Distance_to_Tech_Hub"]   → β₂
# ----------------------------------------------------------
beta_0 = model.params["const"]                   # 截距
beta_1 = model.params["Property_Age"]            # Property_Age 系数
beta_2 = model.params["Distance_to_Tech_Hub"]    # Distance_to_Tech_Hub 系数

print(f"\n截距 β₀ = {beta_0:,.2f}")
print(f"Property_Age 系数 β₁ = {beta_1:,.2f}")
print(f"Distance_to_Tech_Hub 系数 β₂ = {beta_2:,.2f}")

# ----------------------------------------------------------
# STEP 4: 生成 Meshgrid 用于绘制回归超平面
#
# np.linspace 在两个自变量的值域内各取 50 个等间距点
# np.meshgrid 将两个 1D 数组扩展为 2D 网格矩阵：
#   age_grid[i,j]  = 第 j 个 Property_Age 值
#   dist_grid[i,j] = 第 i 个 Distance_to_Tech_Hub 值
# 这样网格上每个 (i,j) 点都对应一个 (age, dist) 组合
# ----------------------------------------------------------
age_range  = np.linspace(df["Property_Age"].min(),
                         df["Property_Age"].max(), 50)
dist_range = np.linspace(df["Distance_to_Tech_Hub"].min(),
                         df["Distance_to_Tech_Hub"].max(), 50)

age_grid, dist_grid = np.meshgrid(age_range, dist_range)

# 用回归方程在网格每个点上计算预测值：
# Ŷ = β₀ + β₁·Age + β₂·Distance
price_grid = beta_0 + beta_1 * age_grid + beta_2 * dist_grid

# ----------------------------------------------------------
# STEP 5: 用 plotly.graph_objects 绘制 3D 交互图
# ----------------------------------------------------------
fig = go.Figure()

# --- 5a. 散点图：真实数据点 ---
fig.add_trace(go.Scatter3d(
    x=df["Property_Age"],
    y=df["Distance_to_Tech_Hub"],
    z=df["Sale_Price"],
    mode="markers",
    marker=dict(
        size=4,
        color=df["Sale_Price"],      # 按价格着色
        colorscale="Viridis",
        opacity=0.75,
        colorbar=dict(title="Sale Price ($)", x=1.05)
    ),
    name="Actual Data Points",
    hovertemplate=(
        "Property Age: %{x:.1f} yrs<br>"
        "Distance: %{y:.1f} km<br>"
        "Sale Price: $%{z:,.0f}<extra></extra>"
    )
))

# --- 5b. 超平面：OLS 拟合回归面 ---
fig.add_trace(go.Surface(
    x=age_grid,
    y=dist_grid,
    z=price_grid,
    colorscale="RdBu",
    opacity=0.55,
    name="Regression Hyperplane",
    showscale=False,
    hovertemplate=(
        "Property Age: %{x:.1f} yrs<br>"
        "Distance: %{y:.1f} km<br>"
        "Predicted Price: $%{z:,.0f}<extra></extra>"
    )
))

# --- 5c. 布局与标注 ---
r2   = model.rsquared
rmse = np.sqrt(model.mse_resid)

fig.update_layout(
    title=dict(
        text=(f"3D OLS Regression Hyperplane<br>"
              f"<sup>Sale_Price ~ Property_Age + Distance_to_Tech_Hub | "
              f"R² = {r2:.3f} | RMSE = ${rmse:,.0f}</sup>"),
        x=0.5
    ),
    scene=dict(
        xaxis_title="Property Age (years)",
        yaxis_title="Distance to Tech Hub (km)",
        zaxis_title="Sale Price ($)",
        xaxis=dict(backgroundcolor="rgb(240,240,255)"),
        yaxis=dict(backgroundcolor="rgb(240,255,240)"),
        zaxis=dict(backgroundcolor="rgb(255,245,240)"),
    ),
    legend=dict(x=0.01, y=0.95),
    margin=dict(l=0, r=0, t=80, b=0),
    height=700
)

fig.show()

# ----------------------------------------------------------
# STEP 6: 解读说明（控制混淆变量）
# ----------------------------------------------------------
print("\n" + "="*60)
print("📊 如何解读这个 3D 超平面（控制混淆变量）")
print("="*60)
print(f"""
回归方程：
  Sale_Price = {beta_0:,.0f}
             + ({beta_1:,.0f}) × Property_Age
             + ({beta_2:,.0f}) × Distance_to_Tech_Hub

【β₁ = {beta_1:,.0f}】
  在 Distance_to_Tech_Hub【保持不变】的前提下，
  Property_Age 每增加 1 年，Sale_Price 变化 {beta_1:,.0f} 元。
  → 这已"控制"了距离的影响，剥离了距离对价格的干扰。

【β₂ = {beta_2:,.0f}】
  在 Property_Age【保持不变】的前提下，
  Distance_to_Tech_Hub 每增加 1 公里，Sale_Price 变化 {beta_2:,.0f} 元。
  → 这已"控制"了房龄的影响，剥离了房龄对价格的干扰。

【超平面的意义】
  平面上每一点代表：给定 (Property_Age, Distance) 时的预测均值。
  数据点与平面的垂直距离 = 残差（模型无法解释的部分）。
  平面的倾斜方向与坡度直观体现了每个变量的独立边际效应。

  R² = {r2:.3f} 表示模型解释了因变量 {r2*100:.1f}% 的方差。
""")

                            OLS Regression Results                            
Dep. Variable:             Sale_Price   R-squared:                       0.947
Model:                            OLS   Adj. R-squared:                  0.946
Method:                 Least Squares   F-statistic:                     1747.
Date:                Fri, 13 Mar 2026   Prob (F-statistic):          4.44e-126
Time:                        18:43:52   Log-Likelihood:                -2342.3
No. Observations:                 200   AIC:                             4691.
Df Residuals:                     197   BIC:                             4701.
Df Model:                           2                                         
Covariance Type:            nonrobust                                         
                           coef    std err          t      P>|t|      [0.025      0.975]
----------------------------------------------------------------------------------------
const                 8.026e+05 


📊 如何解读这个 3D 超平面（控制混淆变量）

回归方程：
  Sale_Price = 802,641
             + (-4,994) × Property_Age
             + (-12,187) × Distance_to_Tech_Hub

【β₁ = -4,994】
  在 Distance_to_Tech_Hub【保持不变】的前提下，
  Property_Age 每增加 1 年，Sale_Price 变化 -4,994 元。
  → 这已"控制"了距离的影响，剥离了距离对价格的干扰。

【β₂ = -12,187】
  在 Property_Age【保持不变】的前提下，
  Distance_to_Tech_Hub 每增加 1 公里，Sale_Price 变化 -12,187 元。
  → 这已"控制"了房龄的影响，剥离了房龄对价格的干扰。

【超平面的意义】
  平面上每一点代表：给定 (Property_Age, Distance) 时的预测均值。
  数据点与平面的垂直距离 = 残差（模型无法解释的部分）。
  平面的倾斜方向与坡度直观体现了每个变量的独立边际效应。

  R² = 0.947 表示模型解释了因变量 94.7% 的方差。



In [32]:
# If plotly is not already available in your Colab runtime, uncomment this:
# !pip install plotly

import pandas as pd
import numpy as np
import statsmodels.formula.api as smf
import plotly.graph_objects as go

# =========================
# Step 1: Load your dataset
# =========================
# Replace the file name below with your actual uploaded file name if needed
file_path = '/content/Zillow_California_2026_Hedonic.csv'
df = pd.read_csv(file_path)

# Quick check
print(df.head())
print(df.columns)

# ==========================================
# Step 2: Fit the multivariate OLS regression
# ==========================================
model = smf.ols('Sale_Price ~ Property_Age + Distance_to_Tech_Hub', data=df).fit()

print(model.summary())

# ======================================================
# Step 3: Extract coefficients from the fitted OLS model
# ======================================================
# The fitted plane has the form:
# Sale_Price = intercept + b1*(Property_Age) + b2*(Distance_to_Tech_Hub)

intercept = model.params['Intercept']
b_age = model.params['Property_Age']
b_distance = model.params['Distance_to_Tech_Hub']

print("\nIntercept:", intercept)
print("Property_Age coefficient:", b_age)
print("Distance_to_Tech_Hub coefficient:", b_distance)

# ============================================================
# Step 4: Create a meshgrid for the regression surface (plane)
# ============================================================
# We generate a grid over the observed range of the two X variables.
# This allows us to compute predicted Sale_Price values at many points
# and draw the fitted hyperplane as a smooth surface.

age_range = np.linspace(df['Property_Age'].min(), df['Property_Age'].max(), 40)
distance_range = np.linspace(df['Distance_to_Tech_Hub'].min(), df['Distance_to_Tech_Hub'].max(), 40)

# Meshgrid creates every combination of age and distance values
age_grid, distance_grid = np.meshgrid(age_range, distance_range)

# ======================================================
# Step 5: Compute predicted values over the full grid
# ======================================================
# Using the regression equation:
# z = intercept + b_age * age_grid + b_distance * distance_grid

price_grid = intercept + b_age * age_grid + b_distance * distance_grid

# ============================================
# Step 6: Build the interactive 3D visualization
# ============================================
fig = go.Figure()

# Add the observed data as a 3D scatter plot
fig.add_trace(
    go.Scatter3d(
        x=df['Property_Age'],
        y=df['Distance_to_Tech_Hub'],
        z=df['Sale_Price'],
        mode='markers',
        marker=dict(
            size=4,
            opacity=0.8
        ),
        name='Observed Data'
    )
)

# Add the fitted regression hyperplane as a surface
fig.add_trace(
    go.Surface(
        x=age_grid,
        y=distance_grid,
        z=price_grid,
        opacity=0.6,
        name='Fitted Regression Plane',
        showscale=False
    )
)

# ============================================
# Step 7: Improve layout and axis labeling
# ============================================
fig.update_layout(
    title='3D Hedonic Pricing Model: Observed Data and Fitted Regression Hyperplane',
    scene=dict(
        xaxis_title='Property_Age',
        yaxis_title='Distance_to_Tech_Hub',
        zaxis_title='Sale_Price'
    ),
    width=1000,
    height=800
)

fig.show()

   Property_Age  Distance_to_Tech_Hub  Sale_Price
0          77.5                  38.1   684100.56
1          11.0                  95.1   413634.22
2          47.7                  73.5   456709.35
3          61.9                  60.3   624533.95
4         100.8                  16.4   870137.54
Index(['Property_Age', 'Distance_to_Tech_Hub', 'Sale_Price'], dtype='object')
                            OLS Regression Results                            
Dep. Variable:             Sale_Price   R-squared:                       0.954
Model:                            OLS   Adj. R-squared:                  0.954
Method:                 Least Squares   F-statistic:                 1.040e+04
Date:                Fri, 13 Mar 2026   Prob (F-statistic):               0.00
Time:                        18:50:15   Log-Likelihood:                -11982.
No. Observations:                1000   AIC:                         2.397e+04
Df Residuals:                     997   BIC:                         